## Завдання 2
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;

In [2]:
import urllib.request
import os
import glob
from datetime import datetime

data_dir = "vhi_data"
os.makedirs(data_dir, exist_ok=True)

def download_vhi(province_id):
    pattern = os.path.join(data_dir, f"VHI_{province_id}_*.csv")
    existing_files = glob.glob(pattern)

    if existing_files:
        print(f"Файл для області {province_id} вже існує ({os.path.basename(existing_files[0])}). Пропускаємо завантаження.")
        return 
    
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"VHI_{province_id}_{timestamp}.csv"
    filepath = os.path.join(data_dir, filename)

    print(f"Завантаження даних для області {province_id}...")
    try:
        urllib.request.urlretrieve(url, filepath)
        print(f"Успішно завантажено: {filename}")
    except Exception as e:
        print(f"Помилка завантаження області {province_id}: {e}")

for i in range(1, 28):
    download_vhi(i)

Файл для області 1 вже існує (VHI_1_20260316_183555.csv). Пропускаємо завантаження.
Файл для області 2 вже існує (VHI_2_20260316_183556.csv). Пропускаємо завантаження.
Файл для області 3 вже існує (VHI_3_20260316_183556.csv). Пропускаємо завантаження.
Файл для області 4 вже існує (VHI_4_20260316_183557.csv). Пропускаємо завантаження.
Файл для області 5 вже існує (VHI_5_20260316_183558.csv). Пропускаємо завантаження.
Файл для області 6 вже існує (VHI_6_20260316_183559.csv). Пропускаємо завантаження.
Файл для області 7 вже існує (VHI_7_20260316_183600.csv). Пропускаємо завантаження.
Файл для області 8 вже існує (VHI_8_20260316_183602.csv). Пропускаємо завантаження.
Файл для області 9 вже існує (VHI_9_20260316_183603.csv). Пропускаємо завантаження.
Файл для області 10 вже існує (VHI_10_20260316_183603.csv). Пропускаємо завантаження.
Файл для області 11 вже існує (VHI_11_20260316_183604.csv). Пропускаємо завантаження.
Файл для області 12 вже існує (VHI_12_20260316_183605.csv). Пропускаємо 

## Завдання 3
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [3]:
import pandas as pd
import os
import glob

def create_vhi_dataframe(data_dir="vhi_data"):
    province_mapping = {
        1: (22, 'Черкаська'), 2: (24, 'Чернігівська'), 3: (23, 'Чернівецька'), 4: (25, 'Республіка Крим'),
        5: (3, 'Дніпропетровська'), 6: (4, 'Донецька'), 7: (8, 'Івано-Франківська'), 8: (19, 'Харківська'),
        9: (20, 'Херсонська'), 10: (21, 'Хмельницька'), 11: (9, 'Київська'), 12: (26, 'м. Київ'),
        13: (10, 'Кіровоградська'), 14: (11, 'Луганська'), 15: (12, 'Львівська'), 16: (13, 'Миколаївська'),
        17: (14, 'Одеська'), 18: (15, 'Полтавська'), 19: (16, 'Рівненська'), 20: (27, 'Севастополь'),
        21: (17, 'Сумська'), 22: (18, 'Тернопільська'), 23: (6, 'Закарпатська'), 24: (1, 'Вінницька'),
        25: (2, 'Волинська'), 26: (7, 'Запорізька'), 27: (5, 'Житомирська')
    }

    all_files = glob.glob(os.path.join(data_dir, "VHI_*.csv"))
    df_list = []

    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']

    for file in all_files:
        filename = os.path.basename(file)
        old_id = int(filename.split('_')[1])

        df = pd.read_csv(file, header=1, names=headers, skipfooter=1, engine='python')

        if 'empty' in df.columns:
            df = df.drop(columns=['empty'])

        df['Year'] = df['Year'].astype(str).str.replace(r'<[^>]+>', '', regex=True)

        df = df[df['Year'].str.strip().str.isnumeric()]
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)

        df = df.dropna()
        df = df[df['VHI'] != -1.0]

        new_id, province_name = province_mapping.get(old_id, (old_id, 'Невідомо'))
        df['Province_ID'] = new_id
        df['Province_Name'] = province_name

        df_list.append(df)

    full_df = pd.concat(df_list, ignore_index=True)

    full_df = full_df[['Year', 'Week', 'Province_ID', 'Province_Name', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']]

    return full_df

df_vhi = create_vhi_dataframe()
print("Дані успішно зчитано та очищено!")
display(df_vhi.head())

Дані успішно зчитано та очищено!


,Year,Week,Province_ID,Province_Name,SMN,SMT,VCI,TCI,VHI
0,1982,1,21,Хмельницька,0.059,258.24,51.11,48.78,49.95
1,1982,2,21,Хмельницька,0.063,261.53,55.89,38.20,47.04
2,1982,3,21,Хмельницька,0.063,263.45,57.30,32.69,44.99
3,1982,4,21,Хмельницька,0.061,265.10,53.96,28.62,41.29
4,1982,5,21,Хмельницька,0.058,266.42,46.87,28.57,37.72


## Завдання 4
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 


In [4]:
def change_province_indices(df):
    index_mapping = {
        1: 22,  # Черкаська
        2: 24,  # Чернігівська
        3: 23,  # Чернівецька
        4: 25,  # Республіка Крим
        5: 3,   # Дніпропетровська
        6: 4,   # Донецька
        7: 8,   # Івано-Франківська
        8: 19,  # Харківська
        9: 20,  # Херсонська
        10: 21, # Хмельницька
        11: 9,  # Київська
        12: 26, # м. Київ
        13: 10, # Кіровоградська
        14: 11, # Луганська
        15: 12, # Львівська
        16: 13, # Миколаївська
        17: 14, # Одеська
        18: 15, # Полтавська
        19: 16, # Рівненська
        20: 27, # Севастополь
        21: 17, # Сумська
        22: 18, # Тернопільська
        23: 6,  # Закарпатська
        24: 1,  # Вінницька
        25: 2,  # Волинська
        26: 7,  # Запорізька
        27: 5   # Житомирська
    }
    
    df_reindexed = df.copy()
    
    df_reindexed['Province_ID'] = df_reindexed['Province_ID'].map(index_mapping)
    
    return df_reindexed

df_vhi_updated = change_province_indices(df_vhi)
print("Індекси успішно змінено!")
display(df_vhi_updated.head())

Індекси успішно змінено!


,Year,Week,Province_ID,Province_Name,SMN,SMT,VCI,TCI,VHI
0,1982,1,17,Хмельницька,0.059,258.24,51.11,48.78,49.95
1,1982,2,17,Хмельницька,0.063,261.53,55.89,38.20,47.04
2,1982,3,17,Хмельницька,0.063,263.45,57.30,32.69,44.99
3,1982,4,17,Хмельницька,0.061,265.10,53.96,28.62,41.29
4,1982,5,17,Хмельницька,0.058,266.42,46.87,28.57,37.72


## Завдання 5
Реалізувати процедури для формування вибірок 

**5.1**
Ряд VHI для області за вказаний рік;

In [5]:
def get_vhi_series_by_year(df, province_id, year):
    filtered_df = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    
    result = filtered_df[['Week', 'VHI']]
    
    return result

print("VHI для Вінницької області (ID=1) за 2020 рік:")
display(get_vhi_series_by_year(df_vhi_updated, province_id=1, year=2020).head())

VHI для Вінницької області (ID=1) за 2020 рік:


,Week,VHI
43460,1,38.33
43461,2,37.57
43462,3,37.48
43463,4,40.47
43464,5,44.28


**5.2** Ряд VHI за вказаний діапазон років для вказаних областей

In [6]:
def get_vhi_by_year_range(df, province_ids, start_year, end_year):
    filtered_df = df[(df['Province_ID'].isin(province_ids)) & 
                     (df['Year'] >= start_year) & 
                     (df['Year'] <= end_year)]
    
    result = filtered_df[['Year', 'Week', 'Province_ID', 'Province_Name', 'VHI']]
    
    return result

print("VHI для областей 1 та 2 за 2015-2017 роки:")
display(get_vhi_by_year_range(df_vhi_updated, province_ids=[1, 2], start_year=2015, end_year=2017).head(10))

VHI для областей 1 та 2 за 2015-2017 роки:


,Year,Week,Province_ID,Province_Name,VHI
43200,2015,1,1,Чернігівська,50.69
43201,2015,2,1,Чернігівська,50.77
43202,2015,3,1,Чернігівська,48.43
43203,2015,4,1,Чернігівська,47.89
43204,2015,5,1,Чернігівська,47.91
43205,2015,6,1,Чернігівська,46.76
43206,2015,7,1,Чернігівська,44.75
43207,2015,8,1,Чернігівська,41.95
43208,2015,9,1,Чернігівська,39.44
43209,2015,10,1,Чернігівська,37.94


**5.3** Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [7]:
def get_vhi_extremes_and_stats(df, province_ids, years):
    filtered_df = df[(df['Province_ID'].isin(province_ids)) & (df['Year'].isin(years))]

    if filtered_df.empty:
        return "Дані відсутні для вказаних критеріїв."

    stats_df = pd.DataFrame({
        'Метрика': ['Мінімум (min)', 'Максимум (max)', 'Середнє (mean)', 'Медіана (median)'],
        'Значення VHI': [
            filtered_df['VHI'].min(),
            filtered_df['VHI'].max(),
            round(filtered_df['VHI'].mean(), 2), 
            filtered_df['VHI'].median()
        ]
    })

    return stats_df

print("Статистика VHI для областей 1 та 3 за 2005 і 2010 роки:")
display(get_vhi_extremes_and_stats(df_vhi_updated, province_ids=[1, 3], years=[2006, 2010]))

Статистика VHI для областей 1 та 3 за 2005 і 2010 роки:


,Метрика,Значення VHI
0,Мінімум (min),21.83
1,Максимум (max),69.80
2,Середнє (mean),47.88
3,Медіана (median),46.75
